In [1]:
import numpy as np
import pandas as pd
import os

from google.colab import files

uploaded = files.upload()

df = pd.read_csv("Day11_Messy_Company_Employee_Dataset.csv")
print("Shape:", df.shape)
print("Columns & Types:")
print(df.info())
print("\nFirst 10 rows:")
print(df.head(10))

Saving Day11_Messy_Company_Employee_Dataset.csv to Day11_Messy_Company_Employee_Dataset.csv
Shape: (157, 12)
Columns & Types:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 157 entries, 0 to 156
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Employee_ID        157 non-null    object 
 1   Employee_Name      157 non-null    object 
 2   Department         152 non-null    object 
 3   Job_Title          157 non-null    object 
 4   Age                153 non-null    float64
 5   Gender             152 non-null    object 
 6   Annual_Salary      152 non-null    float64
 7   Experience_Years   154 non-null    float64
 8   Joining_Date       157 non-null    object 
 9   City               152 non-null    object 
 10  Performance_Score  154 non-null    float64
 11  Work_Mode          155 non-null    object 
dtypes: float64(4), object(8)
memory usage: 14.8+ KB
None

First 10 rows:
  Employee_ID Employee_

In [2]:
print("--- Missing Values ---")
print(df.isnull().sum())

print("\n--- Duplicates ---")
print("Full duplicates:", df.duplicated().sum())
print("Duplicates by Employee_ID:", df.duplicated(subset=['Employee_ID']).sum())

print("\n--- Unique values in categorical columns ---")
for col in ['Department', 'Job_Title', 'Gender', 'City', 'Work_Mode']:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False))

print("\n--- Numeric summary ---")
print(df.describe())

--- Missing Values ---
Employee_ID          0
Employee_Name        0
Department           5
Job_Title            0
Age                  4
Gender               5
Annual_Salary        5
Experience_Years     3
Joining_Date         0
City                 5
Performance_Score    3
Work_Mode            2
dtype: int64

--- Duplicates ---
Full duplicates: 7
Duplicates by Employee_ID: 7

--- Unique values in categorical columns ---

Department:
Department
Marketing           23
Engineering         21
Customer Success    20
Finance             19
Sales               19
Operations          17
Human Resources     16
Data & Analytics    15
NaN                  5
ENGINEERING          1
engineering          1
Name: count, dtype: int64

Job_Title:
Job_Title
Data Scientist                    13
Senior Software Engineer          11
Marketing Executive               10
Sales Manager                      9
Financial Analyst                  9
Support Specialist                 8
Content Strategist         

In [3]:
print("String columns inspection:")
for col in ['Employee_Name', 'Department', 'Job_Title', 'Gender', 'City', 'Work_Mode']:
    print(col, [x for x in df[col].unique() if isinstance(x, str) and (x != x.strip() or not x.istitle())])

print("\nJob Title vs Department mapping:")
print(df.groupby(['Job_Title', 'Department'], dropna=False).size())

String columns inspection:
Employee_Name []
Department ['ENGINEERING', 'engineering']
Job_Title ['HR Manager', 'QA Engineer', 'BI Analyst', 'HR Executive']
Gender ['female', 'MALE']
City ['delhi', 'DELHI', 'Delhi ']
Work_Mode ['REMOTE', 'remote']

Job Title vs Department mapping:
Job_Title                       Department      
Accountant                      Finance              6
BI Analyst                      Data & Analytics     3
Business Development Executive  Sales                5
                                NaN                  2
Content Strategist              Marketing            8
Customer Success Executive      Customer Success     5
                                NaN                  1
Customer Success Manager        Customer Success     7
Data Scientist                  Data & Analytics    12
                                NaN                  1
Finance Manager                 Finance              4
Financial Analyst               Finance              9
HR Executi

In [4]:
print("Missing Department rows:")
print(df[df['Department'].isna() | df['Department'].str.contains('engineering|ENGINEERING', case=False, na=False)][['Employee_ID', 'Job_Title', 'Department']])

Missing Department rows:
    Employee_ID                       Job_Title   Department
0       EMP0098                  Data Scientist          NaN
3       EMP0105               Software Engineer  Engineering
7       EMP0026                     QA Engineer  Engineering
27      EMP0107        Senior Software Engineer  Engineering
32      EMP0035        Senior Software Engineer  Engineering
38      EMP0131        Senior Software Engineer  Engineering
41      EMP0091                     QA Engineer  Engineering
42      EMP0069                     QA Engineer  Engineering
45      EMP0059                     QA Engineer  Engineering
46      EMP0071               Software Engineer  Engineering
50      EMP0146        Senior Software Engineer  Engineering
51      EMP0007        Senior Software Engineer  Engineering
57      EMP0134  Business Development Executive          NaN
59      EMP0034        Senior Software Engineer  Engineering
66      EMP0130        Senior Software Engineer  Engineering

In [5]:
duplicates_mask = df.duplicated()
print(f"Number of duplicate rows: {duplicates_mask.sum()}")
print("Duplicate rows:")
print(df[df.duplicated(keep=False)].sort_values('Employee_ID'))

df_clean = df.drop_duplicates().copy()
print(f"Shape after drop_duplicates: {df_clean.shape}")

Number of duplicate rows: 7
Duplicate rows:
    Employee_ID  Employee_Name  Department             Job_Title   Age  \
70      EMP0021      Riya Khan  Operations  Operations Executive  49.0   
104     EMP0021      Riya Khan  Operations  Operations Executive  49.0   
78      EMP0048     Nikhil Dar     Finance     Financial Analyst  27.0   
110     EMP0048     Nikhil Dar     Finance     Financial Analyst  27.0   
11      EMP0075    Manya Joshi       Sales         Sales Manager  29.0   
61      EMP0075    Manya Joshi       Sales         Sales Manager  29.0   
25      EMP0097      Zoya Shah     Finance     Financial Analyst  41.0   
55      EMP0097      Zoya Shah     Finance     Financial Analyst  41.0   
92      EMP0120   Saira Sheikh   Marketing   Marketing Executive  52.0   
156     EMP0120   Saira Sheikh   Marketing   Marketing Executive  52.0   
85      EMP0139  Simran Sharma     Finance            Accountant  32.0   
100     EMP0139  Simran Sharma     Finance            Accountant  32

In [6]:
print(df_clean.isnull().sum())

Employee_ID          0
Employee_Name        0
Department           5
Job_Title            0
Age                  4
Gender               5
Annual_Salary        4
Experience_Years     3
Joining_Date         0
City                 5
Performance_Score    3
Work_Mode            2
dtype: int64


In [8]:
job_to_dept = {
    'Data Scientist': 'Data & Analytics',
    'BI Analyst': 'Data & Analytics',
    'Senior Software Engineer': 'Engineering',
    'QA Engineer': 'Engineering',
    'Software Engineer': 'Engineering',
    'Accountant': 'Finance',
    'Finance Manager': 'Finance',
    'Financial Analyst': 'Finance',
    'HR Executive': 'Human Resources',
    'HR Manager': 'Human Resources',
    'Recruiter': 'Human Resources',
    'Marketing Executive': 'Marketing',
    'Marketing Manager': 'Marketing',
    'Content Strategist': 'Marketing',
    'Operations Executive': 'Operations',
    'Operations Manager': 'Operations',
    'Process Analyst': 'Operations',
    'Sales Executive': 'Sales',
    'Sales Manager': 'Sales',
    'Business Development Executive': 'Sales',
    'Customer Success Executive': 'Customer Success',
    'Customer Success Manager': 'Customer Success',
    'Support Specialist': 'Customer Success'
}

df_clean['Department'] = df_clean['Job_Title'].map(job_to_dept)
print("Department nulls after mapping:", df_clean['Department'].isnull().sum())
print("Unique departments:", df_clean['Department'].unique())
df_clean['Department'] = df_clean['Job_Title'].map(job_to_dept)
print("Department nulls after mapping:", df_clean['Department'].isnull().sum())
print("Unique departments:", df_clean['Department'].unique())

Department nulls after mapping: 0
Unique departments: ['Data & Analytics' 'Marketing' 'Human Resources' 'Engineering' 'Sales'
 'Finance' 'Customer Success' 'Operations']
Department nulls after mapping: 0
Unique departments: ['Data & Analytics' 'Marketing' 'Human Resources' 'Engineering' 'Sales'
 'Finance' 'Customer Success' 'Operations']


In [9]:
print("Gender unique:", df_clean['Gender'].unique())
df_clean['Gender'] = df_clean['Gender'].astype(str).str.strip().str.title().replace({'Nan': np.nan, 'None': np.nan})
print("Gender cleaned unique:", df_clean['Gender'].value_counts(dropna=False))

df_clean['City'] = df_clean['City'].astype(str).str.strip().str.title().replace({'Nan': np.nan, 'None': np.nan})
print("\nCity cleaned unique:", df_clean['City'].value_counts(dropna=False))

df_clean['Work_Mode'] = df_clean['Work_Mode'].astype(str).str.strip().str.title().replace({'Nan': np.nan, 'None': np.nan})
print("\nWork_Mode cleaned unique:", df_clean['Work_Mode'].value_counts(dropna=False))

Gender unique: ['Other' 'Female' 'Male' nan 'female' 'MALE']
Gender cleaned unique: Gender
Male      72
Female    66
Other      7
NaN        5
Name: count, dtype: int64

City cleaned unique: City
Pune          23
Chandigarh    22
Jaipur        17
Hyderabad     14
Delhi         14
Srinagar      13
Mumbai        13
Bengaluru     11
Chennai        9
Kolkata        9
NaN            5
Name: count, dtype: int64

Work_Mode cleaned unique: Work_Mode
Remote    56
Hybrid    48
Office    44
NaN        2
Name: count, dtype: int64


In [10]:
print("Null numeric columns:")
print("Age nulls:", df_clean[df_clean['Age'].isna()][['Employee_ID', 'Job_Title', 'Age', 'Experience_Years']])
print("Annual_Salary nulls:", df_clean[df_clean['Annual_Salary'].isna()][['Employee_ID', 'Job_Title', 'Department', 'Annual_Salary']])
print("Experience_Years nulls:", df_clean[df_clean['Experience_Years'].isna()][['Employee_ID', 'Job_Title', 'Age', 'Experience_Years']])
print("Performance_Score nulls:", df_clean[df_clean['Performance_Score'].isna()][['Employee_ID', 'Performance_Score']])

Null numeric columns:
Age nulls:     Employee_ID                 Job_Title  Age  Experience_Years
32      EMP0035  Senior Software Engineer  NaN               0.9
103     EMP0122        Operations Manager  NaN              12.8
109     EMP0090              HR Executive  NaN              12.1
137     EMP0008       Marketing Executive  NaN               1.3
Annual_Salary nulls:     Employee_ID             Job_Title  Department  Annual_Salary
35      EMP0142     Marketing Manager   Marketing            NaN
63      EMP0093  Operations Executive  Operations            NaN
78      EMP0048     Financial Analyst     Finance            NaN
124     EMP0012     Marketing Manager   Marketing            NaN
Experience_Years nulls:     Employee_ID                 Job_Title   Age  Experience_Years
3       EMP0105         Software Engineer  30.0               NaN
80      EMP0029        Support Specialist  51.0               NaN
128     EMP0058  Senior Software Engineer  52.0               NaN
Performa

In [11]:
df_raw = pd.read_csv('Day11_Messy_Company_Employee_Dataset.csv')

df_cleaned = df_raw.drop_duplicates().copy()

df_cleaned['Gender'] = df_cleaned['Gender'].astype(str).str.strip().str.title().replace({'Nan': np.nan, 'None': np.nan})
df_cleaned['City'] = df_cleaned['City'].astype(str).str.strip().str.title().replace({'Nan': np.nan, 'None': np.nan})
df_cleaned['Work_Mode'] = df_cleaned['Work_Mode'].astype(str).str.strip().str.title().replace({'Nan': np.nan, 'None': np.nan})

df_cleaned['Department'] = df_cleaned['Job_Title'].map(job_to_dept)

gender_mode = df_cleaned['Gender'].mode()[0]
city_mode = df_cleaned['City'].mode()[0]
work_mode_mode = df_cleaned['Work_Mode'].mode()[0]

df_cleaned['Gender'] = df_cleaned['Gender'].fillna(gender_mode)
df_cleaned['City'] = df_cleaned['City'].fillna(city_mode)
df_cleaned['Work_Mode'] = df_cleaned['Work_Mode'].fillna(work_mode_mode)

age_median = df_cleaned['Age'].median()
salary_median = df_cleaned['Annual_Salary'].median()
exp_median = df_cleaned['Experience_Years'].median()
perf_mode = df_cleaned['Performance_Score'].mode()[0]

df_cleaned['Age'] = df_cleaned['Age'].fillna(age_median).astype(int)
df_cleaned['Annual_Salary'] = df_cleaned['Annual_Salary'].fillna(salary_median).round(2)
df_cleaned['Experience_Years'] = df_cleaned['Experience_Years'].fillna(exp_median).round(1)
df_cleaned['Performance_Score'] = df_cleaned['Performance_Score'].fillna(perf_mode).astype(int)

df_cleaned['Joining_Date'] = pd.to_datetime(df_cleaned['Joining_Date']).dt.strftime('%Y-%m-%d')
df_cleaned = df_cleaned.sort_values(by='Employee_ID').reset_index(drop=True)

output_filename = 'Cleaned_Company_Employee_Dataset.csv'
df_cleaned.to_csv(output_filename, index=False)

print(f"Exported to {output_filename} successfully!")
print("Cleaned DataFrame info:")
print(df_cleaned.info())
print("\nCleaned DataFrame missing values:")
print(df_cleaned.isnull().sum())

Exported to Cleaned_Company_Employee_Dataset.csv successfully!
Cleaned DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Employee_ID        150 non-null    object 
 1   Employee_Name      150 non-null    object 
 2   Department         150 non-null    object 
 3   Job_Title          150 non-null    object 
 4   Age                150 non-null    int64  
 5   Gender             150 non-null    object 
 6   Annual_Salary      150 non-null    float64
 7   Experience_Years   150 non-null    float64
 8   Joining_Date       150 non-null    object 
 9   City               150 non-null    object 
 10  Performance_Score  150 non-null    int64  
 11  Work_Mode          150 non-null    object 
dtypes: float64(2), int64(2), object(8)
memory usage: 14.2+ KB
None

Cleaned DataFrame missing values:
Employee_ID          0
Employee_N

In [12]:
comparison = pd.DataFrame({
    'Metric / Column': [
        'Total Records',
        'Duplicate Rows',
        'Department Missing/Inconsistent',
        'Gender Missing/Inconsistent',
        'City Missing/Inconsistent',
        'Work_Mode Missing/Inconsistent',
        'Age Missing (Type)',
        'Annual_Salary Missing',
        'Experience_Years Missing',
        'Performance_Score Missing (Type)',
        'Joining_Date Format'
    ],
    'Before Cleaning': [
        '157 rows',
        '7 duplicate records',
        '5 missing, mixed cases (e.g. ENGINEERING)',
        '5 missing, casing issues (female, MALE)',
        '5 missing, whitespace & cases (Delhi , delhi)',
        '2 missing, casing issues (REMOTE, remote)',
        '4 missing (float64)',
        '5 missing values',
        '3 missing values',
        '3 missing (float64)',
        'String object'
    ],
    'After Cleaning': [
        '150 unique rows',
        '0 duplicates',
        '0 missing, standardized & mapped from Job Title',
        '0 missing, Title Case, mode-imputed',
        '0 missing, trimmed & Title Case, mode-imputed',
        '0 missing, Title Case, mode-imputed',
        '0 missing (int64, median-imputed: 39)',
        '0 missing (float64, median-imputed: $90,046.50)',
        '0 missing (float64, median-imputed: 9.3 yrs)',
        '0 missing (int64, mode-imputed: 4)',
        'Standardized ISO Date (YYYY-MM-DD)'
    ]
})

print(comparison.to_string(index=False))

                 Metric / Column                               Before Cleaning                                  After Cleaning
                   Total Records                                      157 rows                                 150 unique rows
                  Duplicate Rows                           7 duplicate records                                    0 duplicates
 Department Missing/Inconsistent     5 missing, mixed cases (e.g. ENGINEERING) 0 missing, standardized & mapped from Job Title
     Gender Missing/Inconsistent       5 missing, casing issues (female, MALE)             0 missing, Title Case, mode-imputed
       City Missing/Inconsistent 5 missing, whitespace & cases (Delhi , delhi)   0 missing, trimmed & Title Case, mode-imputed
  Work_Mode Missing/Inconsistent     2 missing, casing issues (REMOTE, remote)             0 missing, Title Case, mode-imputed
              Age Missing (Type)                           4 missing (float64)           0 missing (int64, medi